In [ ]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic 
%pip install litellm


In [3]:
import os
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv()


True

In [4]:
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 5
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)


search_tool = TavilySearchTool()


In [5]:
llm = LLM(
    model="gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)


In [6]:
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

# Writer agent
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [7]:
# ---- Tasks ----
task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher
)

task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 200 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer,
    context=[task_research]
)

In [8]:

crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    process=Process.sequential,
    verbose=True
)

result = await crew.kickoff_async()

print("\nFinal Output:\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ed887125-81c3-40e9-89fb-6085c6973ecd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web and identify the top 3 recent advancements in AI for FMCG.                                │
│  ID: baf019c9-31fa-4dc2-b1a2-f5986842b8ab                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Search the web and identify the top 3 recent advancements in AI for FMCG.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#9) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'recent advancements AI for FMCG 2024 2025 top trends generative AI demand forecasting         │
│  personalization supply chain'}                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#10) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': "generative AI demand forecasting FMCG 2024 research case study 'generative models' 'demand    │
│  forecasting' retail FMCG"}                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - Generative AI (LLMs + multimodal)                                                                            │
│    - What’s new: High-quality, context-aware LLMs and image models integrated with internal SKU/sales data and  │
│  MLOps guardrails.                                                                                              │
│    - Use cases: product ideation, packaging mockups, localized creative, conversational commerce and            │
│  regulatory-first drafts.                                                                                       │
│    - Benefit/consideration: Dramatically faster creative throughput and lower localization cost; requires       │
│  human-in-the-loop, brand safety and IP governance.                                                             │
│                                                                                                                 │
│  - Next‑generation demand forecasting & inventory optimization                                                  │
│    - What’s new: Probabilistic and causal ML, temporal models and real-time data fusion tied to end‑to‑end      │
│  replenishment orchestration.                                                                                   │
│    - Use cases: more accurate forecasts, promotion/cannibalization modeling, automated store-level              │
│  replenishment, scenario planning.                                                                              │
│    - Benefit/consideration: Lower safety stocks, fewer OOS and less waste; depends on clean POS/ERP data and    │
│  changes to downstream ordering logic.                                                                          │
│                                                                                                                 │
│  - Computer vision & in‑store shelf analytics                                                                   │
│    - What’s new: SKU‑level CV, edge/mobile deployments and near‑real‑time planogram and OOS detection at        │
│  scale.                                                                                                         │
│    - Use cases: automated retail execution, promo verification, facings/share‑of‑shelf analytics and            │
│  shelf‑level inventory visibility.                                                                              │
│    - Benefit/consideration: Faster corrective action, higher promo compliance and better field efficiency;      │
│  accuracy hinges on image quality, labeled data and retailer agreements.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Write a concise executive summary using the research notes.                                                    │
│                                                                                                                 │
│  Requirements:                                                                                                  │
│  - Maximum 200 words                                                                                            │
│  - Use bullet points                                                                                            │
│  - Focus only on the 3 key advancements                                                                         │
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ed887125-81c3-40e9-89fb-6085c6973ecd                                                                       │
│  Final Output: - Generative AI (LLMs + multimodal)                                                              │
│    - What’s new: High-quality, context-aware LLMs and image models integrated with internal SKU/sales data and  │
│  MLOps guardrails.                                                                                              │
│    - Use cases: product ideation, packaging mockups, localized creative, conversational commerce and            │
│  regulatory-first drafts.                                                                                       │
│    - Benefit/consideration: Dramatically faster creative throughput and lower localization cost; requires       │
│  human-in-the-loop, brand safety and IP governance.                                                             │
│                                                                                                                 │
│  - Next‑generation demand forecasting & inventory optimization                                                  │
│    - What’s new: Probabilistic and causal ML, temporal models and real-time data fusion tied to end‑to‑end      │
│  replenishment orchestration.                                                                                   │
│    - Use cases: more accurate forecasts, promotion/cannibalization modeling, automated store-level              │
│  replenishment, scenario planning.                                                                              │
│    - Benefit/consideration: Lower safety stocks, fewer OOS and less waste; depends on clean POS/ERP data and    │
│  changes to downstream ordering logic.                                                                          │
│                                                                                                                 │
│  - Computer vision & in‑store shelf analytics                                                                   │
│    - What’s new: SKU‑level CV, edge/mobile deployments and near‑real‑time planogram and OOS detection at        │
│  scale.                                                                                                         │
│    - Use cases: automated retail execution, promo verification, facings/share‑of‑shelf analytics and            │
│  shelf‑level inventory visibility.                                                                              │
│    - Benefit/consideration: Faster corrective action, higher promo compliance and better field efficiency;      │
│  accuracy hinges on image quality, labeled data and retailer agreements.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final Output:

- Generative AI (LLMs + multimodal)
  - What’s new: High-quality, context-aware LLMs and image models integrated with internal SKU/sales data and MLOps guardrails.
  - Use cases: product ideation, packaging mockups, localized creative, conversational commerce and regulatory-first drafts.
  - Benefit/consideration: Dramatically faster creative throughput and lower localization cost; requires human-in-the-loop, brand safety and IP governance.

- Next‑generation demand forecasting & inventory optimization
  - What’s new: Probabilistic and causal ML, temporal models and real-time data fusion tied to end‑to‑end replenishment orchestration.
  - Use cases: more accurate forecasts, promotion/cannibalization modeling, automated store-level replenishment, scenario planning.
  - Benefit/consideration: Lower safety stocks, fewer OOS and less waste; depends on clean POS/ERP data and changes to downstream ordering logic.

- Computer vision & in‑store shelf analytics
  - What’s new: S

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯